# TFT Forecaster — Final

v2.1 config with Optuna-optimized hyperparameters hard-coded.
Features: calendar events, `days_after_christmas`, state identifier.
Christmas = 0 post-prediction.

**Run cells top-to-bottom. Training time: ~8-10 min on RTX 5080.**

In [ ]:
# Cell 1: Imports & GPU check
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer, NaNLabelEncoder
from pytorch_forecasting.metrics import MAE

pl.seed_everything(42, workers=True)
print(f'PyTorch : {torch.__version__}')
print(f'Lightning: {pl.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  Device : {torch.cuda.get_device_name(0)}')
    torch.set_float32_matmul_precision('medium')
else:
    print('No GPU detected.')

## 1. Data Preparation

In [ ]:
# Cell 2: Data prep
TRAIN_PATH  = 'train.csv'
EVENTS_PATH = 'calendar_events.csv'
train  = pd.read_csv(TRAIN_PATH, parse_dates=['date'])
events = pd.read_csv(EVENTS_PATH, parse_dates=['date'])
FORECAST_START = pd.Timestamp('2015-10-01')
FORECAST_END   = pd.Timestamp('2015-12-31')
HORIZON = (FORECAST_END - FORECAST_START).days + 1
print(f'Horizon: {HORIZON} days')

events['event'] = events['event'].str.split(',').str[0].str.strip()
events_map = dict(zip(events['date'], events['event']))
STATE_MAP = {'0':'ALL','1':'CA','2':'CA','3':'CA','4':'CA','5':'TX','6':'TX','7':'TX','8':'WI','9':'WI','10':'WI'}
CHRISTMAS_2015 = pd.Timestamp('2015-12-25')

date_min  = train['date'].min()
all_dates = pd.date_range(date_min, FORECAST_END, freq='D')
all_stores = sorted(train['store_id'].unique())
grid = pd.MultiIndex.from_product([all_stores, all_dates], names=['store_id','date']).to_frame(index=False)
df = grid.merge(train[['store_id','date','revenue']], on=['store_id','date'], how='left')

df['dow']        = df['date'].dt.dayofweek.astype(str)
df['month']      = df['date'].dt.month.astype(str)
df['day']        = df['date'].dt.day.astype(np.float32)
df['week']       = df['date'].dt.isocalendar().week.astype(np.float32)
df['is_weekend'] = (df['date'].dt.dayofweek >= 5).astype(np.float32)
df['event']      = df['date'].map(events_map).fillna('none').astype(str)
df['is_event']   = (df['event'] != 'none').astype(np.float32)

def days_to_holiday(date_series, hm, hd):
    years = date_series.dt.year
    holidays = pd.to_datetime(years.astype(str) + f'-{hm:02d}-{hd:02d}', errors='coerce')
    return (holidays - date_series).dt.days.astype(np.float32).clip(lower=-30, upper=60)

df['days_to_christmas']    = days_to_holiday(df['date'], 12, 25)
df['days_to_thanksgiving'] = days_to_holiday(df['date'], 11, 25)

# Days AFTER Christmas (post-holiday recovery)
def days_after_christmas(date_series):
    years = date_series.dt.year
    christmas = pd.to_datetime(years.astype(str) + '-12-25', errors='coerce')
    delta = (date_series - christmas).dt.days.astype(np.float32)
    prior = pd.to_datetime((years - 1).astype(str) + '-12-25', errors='coerce')
    delta_prior = (date_series - prior).dt.days.astype(np.float32)
    result = np.where(delta >= 0, delta, delta_prior)
    return np.clip(result, 0, 30).astype(np.float32)

df['days_after_christmas'] = days_after_christmas(df['date'])

df['time_idx'] = (df['date'] - date_min).dt.days.astype(np.int64)
df['store_id'] = df['store_id'].astype(str)
df['state']    = df['store_id'].map(STATE_MAP)

mask_past = df['date'] < FORECAST_START
df.loc[mask_past, 'revenue'] = df.loc[mask_past].groupby('store_id')['revenue'].transform(lambda s: s.ffill().bfill())
df['revenue'] = df['revenue'].fillna(0.0).astype(np.float32)
print(f'Grid: {len(df):,} rows')

## 2. Build TimeSeriesDataSet

In [ ]:
# Cell 3: Dataset
MAX_ENCODER_LENGTH    = 180
MAX_PREDICTION_LENGTH = HORIZON
training_cutoff = int(df.loc[df['date'] == FORECAST_START - pd.Timedelta(days=1), 'time_idx'].iloc[0])

training = TimeSeriesDataSet(
    df[df['time_idx'] <= training_cutoff],
    time_idx='time_idx', target='revenue', group_ids=['store_id'],
    max_encoder_length=MAX_ENCODER_LENGTH, min_encoder_length=MAX_ENCODER_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH, min_prediction_length=MAX_PREDICTION_LENGTH,
    static_categoricals=['store_id', 'state'],
    time_varying_known_categoricals=['event', 'dow', 'month'],
    time_varying_known_reals=['time_idx','is_event','is_weekend','day','week',
                              'days_to_christmas','days_to_thanksgiving','days_after_christmas'],
    time_varying_unknown_reals=['revenue'],
    target_normalizer=GroupNormalizer(groups=['store_id'], transformation='softplus'),
    categorical_encoders={'event':NaNLabelEncoder(add_nan=True),'dow':NaNLabelEncoder(add_nan=True),
                          'month':NaNLabelEncoder(add_nan=True),'state':NaNLabelEncoder(add_nan=True)},
    add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
    allow_missing_timesteps=False,
)
validation = TimeSeriesDataSet.from_dataset(training, df, predict=True, stop_randomization=True)

BATCH_SIZE = 128
train_dataloader = training.to_dataloader(train=True,  batch_size=BATCH_SIZE, num_workers=0)
val_dataloader   = validation.to_dataloader(train=False, batch_size=BATCH_SIZE, num_workers=0)
print(f'Training samples: {len(training):,}')

## 3. Train TFT

Optuna-optimized hyperparameters (from prior search):
- `learning_rate = 0.000611`
- `hidden_size = 96`
- `dropout = 0.25`
- `attention_head_size = 4`

In [ ]:
# Cell 4: Train with best known hyperparameters
early_stop = EarlyStopping(monitor='val_loss', min_delta=1e-4, patience=8, mode='min')
lr_monitor = LearningRateMonitor(logging_interval='epoch')

trainer = pl.Trainer(
    max_epochs=30, accelerator='auto', devices=1,
    gradient_clip_val=0.1, callbacks=[early_stop, lr_monitor],
    enable_progress_bar=True, log_every_n_steps=20,
)

tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.000611,
    hidden_size=96,
    attention_head_size=4,
    dropout=0.25,
    hidden_continuous_size=32,
    output_size=1,
    loss=MAE(),
    log_interval=20,
    reduce_on_plateau_patience=3,
)
print(f'TFT parameters: {sum(p.numel() for p in tft.parameters()):,}')

trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

## 4. Predict

In [ ]:
# Cell 5: Predict
best_ckpt = trainer.checkpoint_callback.best_model_path if trainer.checkpoint_callback else ''
if best_ckpt:
    print(f'Loading: {best_ckpt}')
    best_tft = TemporalFusionTransformer.load_from_checkpoint(best_ckpt)
else:
    best_tft = tft

out = best_tft.predict(val_dataloader, mode='prediction', return_x=True, return_index=True)
preds = out.output.cpu().numpy()
if preds.ndim == 3: preds = preds.squeeze(-1)
index = out.index.reset_index(drop=True)
print(f'Predictions shape: {preds.shape}')

## 5. Build Submission

In [ ]:
# Cell 6: Submission
records = []
for i, row in index.iterrows():
    store_id = row['store_id']
    start_idx = int(row['time_idx'])
    for j in range(preds.shape[1]):
        date = date_min + pd.Timedelta(days=start_idx + j)
        records.append({'store_id': store_id, 'date': date, 'prediction': float(preds[i, j])})

submission_long = pd.DataFrame(records)
submission_long['prediction'] = submission_long['prediction'].clip(lower=0)
submission_long.loc[submission_long['date'] == CHRISTMAS_2015, 'prediction'] = 0.0

submission = submission_long.copy()
submission['id'] = submission['store_id'].astype(str) + '_' + submission['date'].dt.strftime('%Y%m%d')
submission = submission[['id', 'prediction']]
print(f'Rows: {len(submission)}')

## 6. Plots

In [ ]:
# Cell 7: Plot
RESULTS_DIR = 'Results'
os.makedirs(RESULTS_DIR, exist_ok=True)
n = len(all_stores)
fig, axes = plt.subplots(n, 1, figsize=(14, 2.5*n), sharex=False)
if n == 1: axes = [axes]
history_window = pd.Timedelta(days=365)
for ax, store_id in zip(axes, all_stores):
    s = str(store_id)
    hist = df[(df['store_id']==s)&(df['date']<FORECAST_START)&(df['date']>=FORECAST_START-history_window)]
    ax.plot(hist['date'], hist['revenue'], color='steelblue', alpha=0.6, label='History')
    pred = submission_long[submission_long['store_id']==s].sort_values('date')
    ax.plot(pred['date'], pred['prediction'], color='darkorange', linewidth=2, label='Forecast')
    ax.axvline(FORECAST_START, linestyle='--', color='gray', alpha=0.7)
    ax.set_title(f'Store {store_id} ({STATE_MAP.get(s,"")})')
    ax.set_ylabel('Revenue'); ax.grid(True, alpha=0.3); ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR,'forecast_plots.png'), dpi=120, bbox_inches='tight')
plt.show()

## 7. Export

In [ ]:
# Cell 8: Export
submission.to_csv(os.path.join(RESULTS_DIR,'forecast_submission.csv'), index=False)
submission_long.to_csv(os.path.join(RESULTS_DIR,'forecast_predictions_long.csv'), index=False)
print('Done.')